This Notebook preprocesses the .csv file of friends for the text mining course

Step 1: Importing CSV file

In [ ]:
import pandas as pd 

try:
    df = pd.read_csv('Friends.csv')
    df = df.astype("string")
except FileNotFoundError:
    print("File not found.")
except Exception as e:
    print(f"An error occurred: {e}")
    
print("File read successfully.")
print("Headers")
print(df.head())

Step2: remove punctuation and special characters
https://stackoverflow.com/questions/39782418/remove-punctuations-in-pandas

In [ ]:
import string
df_no_punct = df.copy()
extra_characters = 'ˆ‘"”∼'
characters_to_remove = string.punctuation + extra_characters
for col in df.columns:
    df_no_punct[col] = df_no_punct[col].str.translate(str.maketrans("", "", characters_to_remove))
# Test case
test = True
if test == True:
    text = 'Hello!”#$%&()*+,-.:;?@[]\\ˆ ‘{|}∼World. We are testing the preprocessing of the text of the TV show Friends.'
    text_no_punct = text.translate(str.maketrans("", "", characters_to_remove))
    print(text_no_punct)

Step 3: Lower case all the text
https://stackoverflow.com/questions/22245171/how-to-lowercase-a-pandas-dataframe-string-column-if-it-has-missing-values

In [ ]:
df_lower = df_no_punct.copy()
for col in df_lower.columns:
    df_lower[col] = df_lower[col].str.lower()
    
#Test case 
if test == True:
    text_lower = text_no_punct.lower()
    print(text_lower)   


Step 4: tokenize the text 
https://www.geeksforgeeks.org/nlp/tokenize-text-using-nltk-python/

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize

col = "Text"

df_tokenized_sent = df_lower.copy()
df_tokenized_word = df_lower.copy()

df_tokenized_sent[col] = df_lower[col].dropna().apply(sent_tokenize)
df_tokenized_word[col] = df_lower[col].dropna().apply(word_tokenize)

#Test case
if test == True:
    text_sent_tokenized = sent_tokenize(text_lower)
    text_word_tokenized = word_tokenize(text_lower)
    print(text_sent_tokenized)
    print(text_word_tokenized)

Step 5: remove stopwords
https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df_tokenized_word[col] = df_tokenized_word[col].dropna().apply(
        lambda tokens: [token for token in tokens if token not in stop_words]
    )

#Test case
if test == True:
    text_no_stopwords = [token for token in text_word_tokenized if token not in stop_words]
    print(text_no_stopwords)

Step 5: Use porter stemmer to stem the words
https://www.geeksforgeeks.org/nlp/porter-stemmer-technique-in-natural-language-processing/

In [ ]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()

df_tokenized_word[col] = df_tokenized_word[col].dropna().apply(
        lambda tokens: [porter_stemmer.stem(token) for token in tokens]
    )

print(df_tokenized_word.head())

Step 6: Perform POS tagging and NER 
https://spacy.io/usage/linguistic-features

In [ ]:
import spacy
from tqdm.auto import tqdm

nlp = spacy.load("en_core_web_sm")
results = []
text_rows = df_lower[col].dropna()

for row_index, row in tqdm(text_rows.items(), total=len(text_rows)):
    doc = nlp(row)
    
    for token in doc:
        results.append({
            "row_index": row_index,
            "text": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_,
            "dependency": token.dep_,
            "shape": token.shape_,
            "is_alpha": token.is_alpha,
            "is_stop": token.is_stop,
        })